In [1]:
import os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from typing import Optional,Dict,Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported


In [2]:
# 1. 加载同目录下的 .env 文件
load_dotenv()

# 2. 从系统环境变量中读取模型名称（如果没读到，默认用 deepseek-chat）
# 注意：LiteLlm 在底层会自动去寻找 os.environ["DEEPSEEK_API_KEY"]，所以我们甚至不需要手动赋给它！
MODEL_NAME = os.getenv("DEEPSEEK_MODEL", "deepseek/deepseek-chat")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(model=llm.model,
                                messages=[{"role": "user", "content": "你好，请问你准备好开始了吗？"}],
                                tools=[]))
print("🤖 DeepSeek 回复：")

print("\nDeepSeek is ready for use.")

ModelResponse(id='e4d4424c-9873-457b-8ac1-19fef2e58f56', created=1788160213, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='你好！我已经准备好了，随时可以开始。请问有什么可以帮你的吗？无论是解答问题、提供建议，还是聊聊天，我都会尽力协助你！😊', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=34, prompt_tokens=12, total_tokens=46, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=12))
🤖 DeepSeek 回复：

DeepSeek is ready for use.


In [3]:
## 设置AgentCaller

In [4]:
from helper import make_agent_caller

print("Libraries imported")

Libraries imported
Libraries imported


In [5]:
## 定义Sub-Agents的工具

In [6]:
from neo4j_for_adk import graphdb


def hello_world_tool(name: str) -> dict:
    """
    这是一个打招呼的工具。当用户要求你向某人说 Hello 或者打招呼时，请调用此工具。

    Args:
        name (str): 需要打招呼的人的名字。

    Returns:
        dict: 包含打招呼结果的字典。
    """
    # 这里是工具的核心逻辑（你可以把它想象成查数据库、调 API 的地方）
    greeting_message = f"Hello, {name}! 欢迎来到 Google ADK 的世界！"

    # 打印一条信息在控制台，方便我们肉眼观察工具是否真的被偷偷调用了
    print(f"\n[🔧 后台运行] hello_world_tool 被触发了，参数 name={name}")

    cypher_query = f"RETURN 'Hello to you, {name}' AS reply"

    return graphdb.send_query(cypher_query)

In [7]:
def say_goodbye_tool() -> dict:
    """
    这是一个道别/告别工具。当用户要求你向某人说再见、道别或结束对话时，请务必调用此工具。
    Returns:
        dict: 包含道别结果的字典。
    """


    # ==============================================================
    # 💡 进阶：如果你想和 hello 工具一样，通过 Neo4j 数据库来返回，
    # 可以把下面 return 字典的代码删掉，换成这句：
    # cypher_query = f"RETURN 'Goodbye to you, {name}' AS reply"
    # return graphdb.send_query(cypher_query)
    # ==============================================================

    # 默认返回给大模型的标准字典
    cypher_query = f"RETURN 'Goodbye to you' AS farewell"
    return graphdb.send_query(cypher_query)

In [8]:
## 定义Sub-Agents

In [9]:
from pathlib import Path

# 使用当前工作目录（Jupyter Notebook 中通常是启动 Notebook 的目录）
PROMPT_FILE = Path.cwd() / "greeting_subagent_instruction.md"
GREETING_SUBAGENT_INSTRUCTION = PROMPT_FILE.read_text(encoding="utf-8")

greeting_subagent = Agent(
    name="greeting_subagent_v1",
    description="这是一个专职的迎宾专员智能体。它的主要职责是使用专用的工具向新用户发送问候和打招呼。当有欢迎新客人的需求时，请呼叫此 Agent。",
    instruction=GREETING_SUBAGENT_INSTRUCTION,
    model=llm,
    tools=[hello_world_tool]
)

print(f"Agent '{greeting_subagent.name}' created")


Agent 'greeting_subagent_v1' created


In [10]:
# 告别subagent
# 1. 读取道别指令文件（假设文件名是 goodbye_subagent_prompt.md）
PROMPT_FILE = Path.cwd() / "goodbye_subagent_instruction.md"
GOODBYE_SUBAGENT_INSTRUCTION = PROMPT_FILE.read_text(encoding="utf-8")

# 2. 定义道别子代理
farewell_subagent = Agent(
    name="farewell_subagent_v1",
    description="这是一个专职的送别专员智能体。它的主要职责是向用户发送道别和再见。当需要结束对话或送别客人时，请呼叫此 Agent。",
    instruction=GOODBYE_SUBAGENT_INSTRUCTION,  # 直接填入从 .md 读取的字符串
    model=llm,
    tools=[say_goodbye_tool]
    # 如果不需要工具，可以省略 tools 参数；如果需要工具，请添加对应工具
)

print(f"Agent '{farewell_subagent.name}' created")

Agent 'farewell_subagent_v1' created


In [11]:
# 创建根agent

In [12]:
root_agent = Agent(
    name="root_agent",
    description="总调度智能体，根据用户意图将任务委派给迎宾专员或送别专员。",
    instruction="""
你是一个总调度智能体，负责理解用户的意图，并将任务分配给合适的子智能体。

## 可用的子智能体
1. **GreetingAgent**：负责问候、欢迎、打招呼等场景。
2. **GoodbyeAgent**：负责道别、再见、结束对话等场景。

## 调度规则
- 如果用户表达的是问候、欢迎、打招呼等意图（例如说“你好”、“嗨”、“欢迎”），请调用 **GreetingAgent**。
- 如果用户表达的是道别、再见、结束对话等意图（例如说“再见”、“拜拜”、“我要走了”），请调用 **GoodbyeAgent**。
- 如果用户意图不明确，请礼貌地询问用户需要问候还是道别。
- 如果用户请求与问候或道别无关，请说明你只负责这些任务，并引导用户联系其他智能体。

## 输出要求
- 只调用相应的子智能体，不要自己生成问候语或道别语。
- 将子智能体返回的结果原样输出给用户。
""",
    model=llm,
    sub_agents=[greeting_subagent, farewell_subagent]  # 注意参数名是 sub_agents（复数）
)

print(f"Agent '{root_agent.name}' created")

Agent 'root_agent' created


In [13]:
from helper import make_agent_caller

root_agent_caller = await make_agent_caller(root_agent)

async def run_team_conversation():
    await root_agent_caller.chat("你好我是zbx", True)

    await root_agent_caller.chat("谢谢，再见", True)

await run_team_conversation()


>>>👤 用户: 你好我是zbx

🤖 Agent 正在思考并执行中...

  [Event] Author: root_agent, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'agent_name': 'greeting_subagent_v1'
    },
    id='call_00_4SSAPVsKcSYlLem6z8rp0031',
    name='transfer_to_agent'
  )
)] role='model'
  [Event] Author: root_agent, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='call_00_4SSAPVsKcSYlLem6z8rp0031',
    name='transfer_to_agent',
    response={
      'result': None
    }
  )
)] role='user'
  [Event] Author: greeting_subagent_v1, Type: Event, Final: False, Content: parts=[Part(
  text="I'll greet the user using the hello tool."
), Part(
  function_call=FunctionCall(
    args={
      'name': 'zbx'
    },
    id='call_00_v7eJuq6rXokdh6hk7HuI5896',
    name='hello_world_tool'
  )
)] role='model'

[🔧 后台运行] hello_world_tool 被触发了，参数 name=zbx
  [Event] Author: greeting_subagent_v1, Type: Event, Final: False, Content: parts=[Part(
  func

In [14]:
# 具备感知能力的工具

In [15]:
from google.adk.tools.tool_context import ToolContext

def say_hello_stateful(tool_context: ToolContext, name: Optional[str] = None) -> dict:
    """
    向用户问好，并将用户的名字记录到状态 (state) 中。
    参数 name: 用户的名字。如果用户在对话中提到了他们的名字，请传入此参数。
    """
    # 访问工具上下文中的状态字典 (Agent 的记忆)
    state = tool_context.state

    # 1. 如果大模型提取到了用户的名字，将其记录到 state 中
    if name:
        tool_context.state["user_name"] = name
        print("\ntool_context.state['user_name']:", tool_context.state["user_name"])

    cypher_query = f"RETURN 'Hello to you, {name}' AS reply"

    return graphdb.send_query(cypher_query)


In [16]:
def say_goodbye_stateful(tool_context: ToolContext) -> dict:
    """
    向用户告别。此工具不需要任何参数（如用户的名字）。
    如果之前的状态中已经记录了用户的名字，则会使用它来个性化告别。
    """
    # 访问工具上下文中的状态字典（Agent 的记忆）
    state = tool_context.state

    # 尝试从状态中获取之前记录的用户名字
    user_name = state.get("user_name")

    # 根据是否有名字生成不同的告别语
    if user_name:
        farewell_msg = f"Goodbye, {user_name}!"
    else:
        farewell_msg = "Goodbye!"

    # 通过 Cypher 查询返回结果（这里假设 graphdb 已定义）
    cypher_query = f"RETURN '{farewell_msg}' AS reply"
    return graphdb.send_query(cypher_query)

In [17]:
greeting_agent_stateful = Agent(
    name="GreetingAgentStateful",
    description="这是一个专职的迎宾专员智能体，具备状态记忆功能。它的主要职责是使用专用的工具向新用户发送问候，并记录用户的名字以便后续使用。当有欢迎新客人的需求时，请呼叫此 Agent。",
    instruction="你是一个迎宾专员。当用户到来时，使用工具 say_hello_stateful 向用户问好。如果用户提到了自己的名字，请作为参数传递给工具，以便记录到状态中。",
    model=llm,  # 假设已定义
    tools=[say_hello_stateful]
)

In [18]:
farewell_agent_stateful = Agent(
    name="farewell_agent_stateful_v1",
    description="这是一个专职的告别专员智能体。它的主要职责是使用专用的工具向用户发送告别语。当有需要和用户说再见的场景时，请呼叫此 Agent。",
    instruction="你是一个告别专员。当用户需要离开或结束对话时，调用工具 say_farewell_stateful 生成告别语。",
    model=llm,
    tools=[say_goodbye_stateful]
)

In [19]:
root_agent_stateful = Agent(
    name="RootAgent",
    description="这是一个总调度智能体，负责根据用户需求分派任务。当用户到来时，调度迎宾专员；当用户离开时，调度告别专员。",
    instruction="你是一个总调度员。根据用户当前的情境，决定调用哪个子 Agent：如果是欢迎新用户，请调用 GreetingAgentStateful；如果是告别或结束对话，请调用 FarewellAgentStateful。",
    model=llm,
    sub_agents=[greeting_agent_stateful, farewell_agent_stateful]  # 将两个子 Agent 注册为可调度的下属
)

In [20]:
root_stateful_caller = await make_agent_caller(root_agent_stateful)

session = await root_stateful_caller.get_session()

print(f"Initial State: {session.state}")

Initial State: {}


In [21]:
async def run_stateful_conversation():
    await root_stateful_caller.chat("你好，我是zbx")

    await root_stateful_caller.chat("谢谢，再见")

await run_stateful_conversation()

session = await root_stateful_caller.get_session()
print(f"Final State: {session.state}")


>>>👤 用户: 你好，我是zbx

tool_context.state['user_name']: zbx
<<< Agent Response: 你好，zbx！很高兴见到你！👋

我是迎宾专员，欢迎你的到来！有什么可以帮到你的吗？


>>>👤 用户: 谢谢，再见
<<< Agent Response: 再见，zbx！感谢你的到来，祝你一切顺利！👋😊

Final State: {'user_name': 'zbx'}


In [23]:
async def run_interative_conversation():
    while True:
        user_query = input("问我问题吧！（或者输入‘exit’退出）")
        if user_query.lower() == 'exit':
            break
        response = await root_stateful_caller.chat(user_query)
        print(f"Response: {response}")

await run_interative_conversation()
session = await root_stateful_caller.get_session()
print(f"Final State: {session.state}")


>>>👤 用户: 你好
<<< Agent Response: 你好，zbx！

看起来你似乎在重新打招呼，而不是要离开呢。作为告别专员，我主要负责在你要离开时为你道别。

如果你是想重新打招呼或需要其他帮助，我可以帮你转交给迎宾专员或其他更合适的智能体哦。

请问你是要离开，还是需要其他帮助呢？😊

Response: 你好，zbx！

看起来你似乎在重新打招呼，而不是要离开呢。作为告别专员，我主要负责在你要离开时为你道别。

如果你是想重新打招呼或需要其他帮助，我可以帮你转交给迎宾专员或其他更合适的智能体哦。

请问你是要离开，还是需要其他帮助呢？😊

>>>👤 用户: 拜拜
<<< Agent Response: 抱歉，告别服务暂时遇到了一点小问题。不过我还是想好好和你道别：

再见，zbx！👋 感谢你的到来，很高兴见到你！祝你度过愉快的一天，一切顺利！😊🌟

期待下次再见！

Response: 抱歉，告别服务暂时遇到了一点小问题。不过我还是想好好和你道别：

再见，zbx！👋 感谢你的到来，很高兴见到你！祝你度过愉快的一天，一切顺利！😊🌟

期待下次再见！
Final State: {'user_name': 'zbx'}
